In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import random
from collections import defaultdict, Counter
from tqdm import tqdm

import torch
import pandas as pd
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from sklearn.model_selection import train_test_split

from egh_vlm.hallu_dataset import load_hallu_dataset
from egh_vlm.utils import Qwen3ModelBundle, get_pred, load_phd_dataset

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/features_egh_full_context_phd_base_qwen3.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_RATIO = 0.7

dataset = load_phd_dataset(DATASET_PATH, IMG_FOLDER_PATH)
features = load_hallu_dataset(FEATURES_PATH) if os.path.isfile(FEATURES_PATH) else None
print(f'Dataset: {len(dataset)} | Features: {len(features) if features else 0}')

In [ ]:
feature_by_id = {
    fid: (emb, grad, label)
    for fid, emb, grad, label in zip(
        features.ids, features.embs, features.grads, features.labels
    )
}

combined_dataset = []
for item in dataset:
    feat = feature_by_id.get(item['id'])
    if feat is None:
        continue
    emb, grad, f_label = feat
    combined_dataset.append({**item, 'emb': emb, 'grad': grad, 'feature_label': f_label})

train_dataset, test_dataset = train_test_split(
    combined_dataset,
    train_size=TRAIN_RATIO,
    stratify=[it['label'] for it in combined_dataset],
    random_state=42,
)

def build_paired_dataset(ds):
    by_couple = defaultdict(list)
    for it in ds:
        by_couple[it['couple_id']].append(it)
    pairs, skipped = [], {'no_truthful': 0, 'no_hallu': 0}
    for cid, items in by_couple.items():
        truthful = [i for i in items if i['label'] == 0]
        hallu    = [i for i in items if i['label'] == 1]
        if not truthful:
            skipped['no_truthful'] += 1; continue
        if not hallu:
            skipped['no_hallu'] += 1; continue
        for h in hallu:
            t = next(
                (x for x in truthful
                 if x['task'] == h['task'] and x['subject'] == h['subject']
                 and x['hitem'] == h['hitem']),
                truthful[0]
            )
            pairs.append({'couple_id': cid, 'task': h['task'], 'truthful': t, 'hallucinated': h})
    print(f'Paired: {len(pairs)} | skipped: {skipped}')
    return pairs

paired_dataset = build_paired_dataset(train_dataset)

In [ ]:
def pool_features(emb, grad, mode='grad_weighted'):
    if emb.numel() == 0:
        return None
    if mode == 'mean':
        return emb.mean(0)
    if mode == 'grad_weighted':
        w = grad.abs().sum(-1)
        w = w / (w.sum() + 1e-8)
        return (emb * w.unsqueeze(-1)).sum(0)
    if mode == 'grad_dir':
        return grad.mean(0)
    if mode == 'last_token':
        return emb[-1]
    raise ValueError(mode)

def compute_task_steering_vectors(
    paired_dataset, hallu_features,
    pool_mode='grad_weighted', balance_by_gt=True, normalize=True, verbose=True
):
    feat_by_id = {i: (e, g) for i, e, g in
                  zip(hallu_features.ids, hallu_features.embs, hallu_features.grads)}
    bucket = defaultdict(lambda: {0: {'truth': [], 'hallu': []},
                                  1: {'truth': [], 'hallu': []}})
    for p in paired_dataset:
        tid, hid = p['truthful']['id'], p['hallucinated']['id']
        if tid not in feat_by_id or hid not in feat_by_id:
            continue
        vt = pool_features(*feat_by_id[tid], mode=pool_mode)
        vh = pool_features(*feat_by_id[hid], mode=pool_mode)
        if vt is None or vh is None:
            continue
        gt = int(p['hallucinated']['question_gt'])
        if gt not in (0, 1):
            continue
        bucket[p['task']][gt]['truth'].append(vt)
        bucket[p['task']][gt]['hallu'].append(vh)

    steering = {}
    for task, per_gt in bucket.items():
        parts, counts = [], {}
        for gt, b in per_gt.items():
            if not b['truth'] or not b['hallu']:
                counts[gt] = 0; continue
            T = torch.stack(b['truth']).float()
            H = torch.stack(b['hallu']).float()
            parts.append(T.mean(0) - H.mean(0))
            counts[gt] = T.shape[0]
        if not parts:
            continue
        v = parts[0] + parts[1] if (balance_by_gt and len(parts) == 2) else sum(parts)
        raw_norm = v.norm().item()
        if normalize:
            v = v / (v.norm() + 1e-8)
        steering[task] = {'vector': v, 'n_gt0': counts.get(0, 0),
                        'n_gt1': counts.get(1, 0), 'raw_norm': raw_norm}
        if verbose:
            print(f'[{task:12s}] gt0={counts.get(0,0):4d} gt1={counts.get(1,0):4d} raw_norm={raw_norm:.3f}')
    return steering

task_steering = compute_task_steering_vectors(
    paired_dataset, features, pool_mode='grad_weighted', balance_by_gt=True
)

In [ ]:
def get_yes_no_axis(model_bundle, source='unembed',
                    variants=((' yes', ' no'), ('yes', 'no'), ('Yes', 'No'), (' Yes', ' No'))):
    tok = model_bundle.processor.tokenizer
    model = model_bundle.model
    if source == 'input':
        W = model.get_input_embeddings().weight.detach().float().cpu()
    elif source == 'unembed':
        for getter in (lambda m: m.lm_head.weight,
                       lambda m: m.language_model.lm_head.weight,
                       lambda m: m.model.lm_head.weight):
            try:
                W = getter(model).detach().float().cpu(); break
            except AttributeError:
                continue
        else:
            raise AttributeError('lm_head not found')
    else:
        raise ValueError(source)
    diffs = []
    for y, n in variants:
        y_ids = tok.encode(y, add_special_tokens=False)
        n_ids = tok.encode(n, add_special_tokens=False)
        if len(y_ids) == 1 and len(n_ids) == 1:
            diffs.append(W[y_ids[0]] - W[n_ids[0]])
    pol = torch.stack(diffs).mean(0)
    return pol / (pol.norm() + 1e-8)

def orthogonalize_task_steering(task_steering, polarity_axis, normalize=True):
    out = {}
    for task, info in task_steering.items():
        v = info['vector'].float()
        proj = float(v @ polarity_axis)
        v_perp = v - proj * polarity_axis
        new_norm = v_perp.norm().item()
        if normalize:
            v_perp = v_perp / (new_norm + 1e-8)
        out[task] = {**info, 'vector': v_perp,
                     'polarity_projection': proj,
                     'residual_norm': new_norm}
    return out

In [ ]:
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct', dtype=torch.float16, device_map=DEVICE
)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct', max_pixels=1024 * 1024)
model_bundle = Qwen3ModelBundle(model, processor, DEVICE)

PROMPT_TEXT = 'Question: {question}\nAnswer ONLY in this exact format, make sure you add the explanation: yes/no, explanation'

In [ ]:
pol_unembed = get_yes_no_axis(model_bundle, source='unembed')
task_steering_bal = compute_task_steering_vectors(
    paired_dataset, features,
    pool_mode='grad_weighted', balance_by_gt=True, normalize=True, verbose=False
)
task_steering_ortho_un = orthogonalize_task_steering(task_steering_bal, pol_unembed)

In [ ]:
def _find_decoder_layers(model):
    candidates = [
        lambda m: m.model.language_model.layers,
        lambda m: m.language_model.model.layers,
        lambda m: m.language_model.layers,
        lambda m: m.model.layers,
        lambda m: m.transformer.h,
    ]
    for fn in candidates:
        try:
            layers = fn(model)
            if hasattr(layers, '__len__') and len(layers) > 0:
                return layers
        except AttributeError:
            continue
    raise AttributeError('Could not locate decoder layers.')

class SteeringHook:
    def __init__(self, model, layer_idx, vector, alpha=1.0,
                 only_new_tokens=False, alpha_relative=True):
        layers = _find_decoder_layers(model)
        idx = len(layers) + layer_idx if layer_idx < 0 else (layer_idx - 1 if layer_idx > 0 else 0)
        self.idx = idx
        p = next(model.parameters())
        self.vector = vector.to(device=p.device, dtype=p.dtype)
        self.alpha = alpha
        self.only_new_tokens = only_new_tokens
        self.alpha_relative = alpha_relative
        self.handle = layers[idx].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        if self.only_new_tokens and hs.shape[1] != 1:
            return output
        last = hs[:, -1, :]
        h_norm = last.float().norm(dim=-1).mean().item()
        v_norm = self.vector.float().norm().item() + 1e-8
        scale = self.alpha * h_norm / v_norm if self.alpha_relative else self.alpha
        delta = scale * self.vector
        hs = hs.clone()
        hs[:, -1, :] = last + delta.to(last.dtype)
        return (hs,) + output[1:] if isinstance(output, tuple) else hs

    def remove(self):
        self.handle.remove()

@torch.no_grad()
def generate_baseline(item, model_bundle, prompt_text=None, max_new_tokens=64, **gk):
    model, processor, device = model_bundle.model, model_bundle.processor, model_bundle.device
    question = item['question']
    if prompt_text:
        question = prompt_text.format(question=question)
    content = []
    if item.get('image_path'):
        content.append({'type': 'image', 'image': item['image_path']})
    content.append({'type': 'text', 'text': question})
    messages = [{'role': 'user', 'content': content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(device)
    gk.setdefault('max_new_tokens', max_new_tokens)
    out = model.generate(**inputs, **gk)
    return processor.batch_decode(
        out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )[0]

@torch.no_grad()
def generate_with_steering(item, model_bundle, task_steering, prompt_text=None,
                           alpha=0.15, layer_idx=-8, max_new_tokens=64, **gk):
    model, processor, device = model_bundle.model, model_bundle.processor, model_bundle.device
    v = task_steering[item['task']]['vector']
    hook = SteeringHook(model, layer_idx, v, alpha=alpha,
                        only_new_tokens=False, alpha_relative=True)
    try:
        question = item['question']
        if prompt_text:
            question = prompt_text.format(question=question)
        content = []
        if item.get('image_path'):
            content.append({'type': 'image', 'image': item['image_path']})
        content.append({'type': 'text', 'text': question})
        messages = [{'role': 'user', 'content': content}]
        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt',
        ).to(device)
        gk.setdefault('max_new_tokens', max_new_tokens)
        out = model.generate(**inputs, **gk)
        return processor.batch_decode(
            out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )[0]
    finally:
        hook.remove()

@torch.no_grad()
def evaluate_hallucinated(dataset, model_bundle, task_steering=None,
                          alpha=0.15, layer_idx=-8, max_new_tokens=64,
                          prompt_text=None, limit=None):
    hallu_items = [it for it in dataset if it['label'] == 1]
    if limit:
        hallu_items = hallu_items[:limit]
    stats = {'total': len(hallu_items), 'fixed': 0, 'still_hallu': 0,
             'unparseable': 0, 'per_task': {}}
    records = []
    for i, item in enumerate(tqdm(hallu_items, desc='Evaluating')):
        gt = item['question_gt']
        if task_steering is not None and item['task'] in task_steering:
            response = generate_with_steering(
                item, model_bundle, task_steering,
                prompt_text=prompt_text, alpha=alpha, layer_idx=layer_idx,
                max_new_tokens=max_new_tokens, do_sample=False)
        else:
            response = generate_baseline(
                item, model_bundle, prompt_text=prompt_text,
                max_new_tokens=max_new_tokens, do_sample=False)
        pred = get_pred(response)
        outcome = 'unparseable' if pred == 0.5 else ('fixed' if pred == gt else 'still_hallu')
        stats[outcome] += 1
        ts = stats['per_task'].setdefault(item['task'],
            {'total': 0, 'fixed': 0, 'still_hallu': 0, 'unparseable': 0})
        ts['total'] += 1; ts[outcome] += 1
        records.append({
            'id': item['id'], 'task': item['task'], 'question_gt': gt,
            'pred': pred, 'outcome': outcome
        })
    n = max(stats['total'], 1)
    print(f'\n=== Overall (n={stats["total"]}) ===')
    print(f'  fixed:       {stats["fixed"]:5d}  ({stats["fixed"]/n:.1%})')
    print(f'  still hallu: {stats["still_hallu"]:5d}  ({stats["still_hallu"]/n:.1%})')
    print(f'  unparseable: {stats["unparseable"]:5d}  ({stats["unparseable"]/n:.1%})')
    return stats, records

In [ ]:
def build_fixed_size_subset(ds, n_per_label=300, seed=42):
    """Balanced subset: n_per_label truthful (label=0) + n_per_label hallucinated (label=1)."""
    rng = random.Random(seed)
    truth = [it for it in ds if it['label'] == 0]
    hallu = [it for it in ds if it['label'] == 1]
    k_t = min(n_per_label, len(truth))
    k_h = min(n_per_label, len(hallu))
    subset = rng.sample(truth, k_t) + rng.sample(hallu, k_h)
    rng.shuffle(subset)
    print(f'Subset: {len(subset)} ({k_t} truthful + {k_h} hallu)')
    return subset

subset_test = build_fixed_size_subset(test_dataset, n_per_label=300, seed=42)

def _per_gt(recs):
    g = {0: [0, 0], 1: [0, 0]}
    for r in recs:
        gt = int(r['question_gt'])
        if gt in g:
            g[gt][1] += 1
            if r['outcome'] == 'fixed':
                g[gt][0] += 1
    return {k: (v[0] / v[1] if v[1] else 0.0, v[1]) for k, v in g.items()}

In [ ]:
# ------------------------------------------------------------------
# Quick sweep: L=-3 and L=-4 on 300/300 subset, alpha 0.15 and 0.3 only
# ------------------------------------------------------------------
contrast_history = []

for layer in [-3, -4]:
    for a in [0.15, 0.3]:
        stats, recs = evaluate_hallucinated(
            subset_test, model_bundle,
            task_steering=task_steering_ortho_un,
            alpha=a, layer_idx=layer,
            prompt_text=PROMPT_TEXT, max_new_tokens=64,
        )
        g = _per_gt(recs)
        n = max(stats['total'], 1)
        contrast_history.append({
            'layer': layer, 'alpha': a,
            'fixed': stats['fixed'], 'still': stats['still_hallu'], 'unparse': stats['unparseable'],
            'fixed_pct': round(stats['fixed']/n*100, 1),
            'gt0_pct': round(g[0][0]*100, 1), 'n_gt0': g[0][1],
            'gt1_pct': round(g[1][0]*100, 1), 'n_gt1': g[1][1],
        })

contrast_summarize = pd.DataFrame(contrast_history)
contrast_summarize = contrast_summarize[[
    'layer', 'alpha', 'fixed', 'still', 'unparse',
    'fixed_pct', 'gt0_pct', 'gt1_pct', 'n_gt0', 'n_gt1'
]]

print('\n=== contrast_summarize ===')
print(contrast_summarize.to_string(index=False))

CSV_PATH = 'contrast_summarize.csv'
contrast_summarize.to_csv(CSV_PATH, index=False)
print(f'\nSaved to: {CSV_PATH}')